In [9]:
import os
import re
import json
import random
from typing import List, Dict, Tuple
from collections import Counter
import chardet

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.metrics import accuracy_score, f1_score, classification_report
from seqeval.metrics import f1_score as seq_f1_score, classification_report as seq_classification_report
from tqdm import tqdm

# %%
# Cell 2 — TranscriptPreprocessor (same as earlier; keeps preprocessing consistent)
class TranscriptPreprocessor:
    def __init__(self, transcripts_dir: str, processed_dir: str):
        self.transcripts_dir = transcripts_dir
        self.processed_dir = processed_dir
        self.intents = set()
        self.slot_types = set()
        self.medical_keywords = {
            'SYMPTOM': ['pain', 'cough', 'breathless', 'itching', 'nausea', 'vomiting', 'fever',
                        'headache', 'dizziness', 'fatigue', 'swelling', 'rash', 'discharge',
                        'bleeding', 'numbness', 'tingling', 'weakness', 'stiffness'],
            'LOCATION': ['chest', 'abdomen', 'back', 'head', 'neck', 'arm', 'leg', 'knee',
                         'shoulder', 'hip', 'ankle', 'wrist', 'upper', 'lower', 'left', 'right'],
            'DURATION': ['days', 'weeks', 'months', 'years', 'hours', 'minutes', 'ago',
                         'since', 'yesterday', 'morning', 'evening', 'night', 'recently'],
            'SEVERITY': ['severe', 'mild', 'moderate', 'intense', 'slight', 'terrible',
                         'bad', 'worse', 'better', 'improved'],
            'PROGRESSION': ['getting worse', 'improving', 'constant', 'intermittent',
                            'comes and goes', 'fluctuates', 'stable'],
            'MEDICATION': ['ibuprofen', 'paracetamol', 'aspirin', 'antacid', 'antibiotic',
                           'inhaler', 'cream', 'ointment', 'tablet', 'pill'],
            'HISTORY': ['diabetes', 'hypertension', 'asthma', 'allergy', 'surgery',
                        'family history', 'previous', 'diagnosed']
        }
        os.makedirs(processed_dir, exist_ok=True)

    def load_transcripts(self) -> List[Dict]:
        transcripts = []
        for filename in os.listdir(self.transcripts_dir):
            if filename.endswith('.txt'):
                filepath = os.path.join(self.transcripts_dir, filename)
                category = filename[:3]
                transcript_id = filename[:-4]
                with open(filepath, 'rb') as f:
                    encoding = chardet.detect(f.read())['encoding'] or 'utf-8'
                with open(filepath, 'r', encoding=encoding, errors='ignore') as f:
                    content = f.read()
                transcripts.append({'id': transcript_id, 'category': category, 'content': content, 'filepath': filepath})
        return transcripts

    def parse_turns(self, content: str) -> List[Dict]:
        turns = []
        for line in content.strip().split('\n'):
            line = line.strip()
            if not line:
                continue
            if line.startswith('D:'):
                turns.append({'speaker': 'Doctor', 'text': line[2:].strip()})
            elif line.startswith('P:'):
                turns.append({'speaker': 'Patient', 'text': line[2:].strip()})
        return turns

    def normalize_text(self, text: str) -> str:
        return re.sub(r'\s+', ' ', text.lower().strip())

    def tokenize(self, text: str) -> List[str]:
        return re.sub(r'([?.!,;])', r' \1 ', text).split()

    def infer_intent(self, speaker: str, text: str) -> str:
        t = text.lower()
        if speaker == 'Doctor':
            if any(q in t for q in ['what', 'when', 'where', 'how', 'which', 'who']):
                if 'symptom' in t or 'feel' in t or 'problem' in t:
                    return 'Ask_Symptom'
                elif any(word in t for word in ['long', 'when', 'start', 'began']):
                    return 'Ask_Duration'
                elif 'where' in t or 'location' in t:
                    return 'Ask_Location'
                elif any(word in t for word in ['severe', 'bad', 'intense']):
                    return 'Ask_Severity'
                elif 'medication' in t or 'medicine' in t or 'taking' in t:
                    return 'Ask_Medication'
                elif 'history' in t or 'before' in t or 'previous' in t:
                    return 'Ask_History'
                else:
                    return 'Ask_General'
            elif any(word in t for word in ['test', 'examine', 'check']):
                return 'Propose_Examination'
            elif any(word in t for word in ['prescribe', 'recommend', 'suggest']):
                return 'Provide_Treatment'
            return 'Doctor_Other'
        else:
            for slot, keywords in self.medical_keywords.items():
                if any(word in t for word in keywords):
                    return f'Provide_{slot.capitalize()}'
            return 'Patient_Other'

    def extract_slots(self, text: str) -> List[Dict]:
        t = text.lower()
        entities = []
        for slot_type, keywords in self.medical_keywords.items():
            for kw in keywords:
                for match in re.finditer(rf'\b{re.escape(kw)}\w*\b', t):
                    entities.append({'type': slot_type, 'text': match.group(), 'start': match.start(), 'end': match.end()})
        entities.sort(key=lambda x: x['start'])
        non_overlap = []
        for e in entities:
            if not non_overlap or e['start'] >= non_overlap[-1]['end']:
                non_overlap.append(e)
        return non_overlap

    def tokens_to_bio(self, tokens: List[str], text: str, entities: List[Dict]) -> List[str]:
        bio = ['O'] * len(tokens)
        char_to_token = []
        pos = 0
        text_lower = text.lower()
        for i, token in enumerate(tokens):
            start = text_lower.find(token, pos)
            start = start if start != -1 else pos
            end = start + len(token)
            char_to_token.append((start, end, i))
            pos = end
        for e in entities:
            overlap = [i for s, en, i in char_to_token if s < e['end'] and en > e['start']]
            if overlap:
                bio[overlap[0]] = f'B-{e["type"]}'
                for idx in overlap[1:]:
                    bio[idx] = f'I-{e["type"]}'
        return bio

    def process_transcripts(self) -> List[Dict]:
        data = []
        for t in self.load_transcripts():
            for idx, turn in enumerate(self.parse_turns(t['content'])):
                text = turn['text']
                norm_text = self.normalize_text(text)
                tokens = self.tokenize(norm_text)
                intent = self.infer_intent(turn['speaker'], text)
                self.intents.add(intent)
                entities = self.extract_slots(norm_text)
                self.slot_types.update([e['type'] for e in entities])
                bio_tags = self.tokens_to_bio(tokens, norm_text, entities)
                data.append({'transcript_id': t['id'], 'category': t['category'], 'turn_id': idx,
                             'speaker': turn['speaker'], 'original_text': text, 'normalized_text': norm_text,
                             'tokens': tokens, 'intent': intent, 'entities': entities, 'bio_tags': bio_tags})
        return data

    def create_splits(self, data: List[Dict], train_ratio=0.7, val_ratio=0.15, test_ratio=0.15, seed=42) -> Tuple[List[Dict], List[Dict], List[Dict]]:
        ids = list({item['transcript_id'] for item in data})
        random.seed(seed)
        random.shuffle(ids)
        n_train = int(len(ids) * train_ratio)
        n_val = int(len(ids) * val_ratio)
        train_ids, val_ids, test_ids = set(ids[:n_train]), set(ids[n_train:n_train+n_val]), set(ids[n_train+n_val:])
        train_data = [x for x in data if x['transcript_id'] in train_ids]
        val_data = [x for x in data if x['transcript_id'] in val_ids]
        test_data = [x for x in data if x['transcript_id'] in test_ids]
        return train_data, val_data, test_data

    def save_processed_data(self, train_data: List[Dict], val_data: List[Dict], test_data: List[Dict]):
        for name, d in zip(['train', 'val', 'test'], [train_data, val_data, test_data]):
            with open(os.path.join(self.processed_dir, f'{name}.json'), 'w') as f:
                json.dump(d, f, indent=2)
        metadata = {'intents': sorted(list(self.intents)),
                    'slots': ['O'] + [f'{p}-{s}' for s in sorted(self.slot_types) for p in ['B', 'I']],
                    'num_intents': len(self.intents),
                    'num_slots': len(self.slot_types)+1,
                    'train_size': len(train_data),
                    'val_size': len(val_data),
                    'test_size': len(test_data)}
        with open(os.path.join(self.processed_dir, 'metadata.json'), 'w') as f:
            json.dump(metadata, f, indent=2)
        return metadata

    def get_statistics(self, data: List[Dict]) -> Dict:
        return {
            'num_utterances': len(data),
            'num_transcripts': len(set([x['transcript_id'] for x in data])),
            'intent_distribution': Counter([x['intent'] for x in data]),
            'category_distribution': Counter([x['category'] for x in data]),
            'speaker_distribution': Counter([x['speaker'] for x in data]),
            'avg_tokens_per_utterance': sum(len(x['tokens']) for x in data)/len(data) if data else 0,
            'slot_type_counts': Counter([e['type'] for x in data for e in x['entities']])
        }

# %%
# Cell 3 — Transformer Dataset and Model
class TransformerSLUDataset(Dataset):
    def __init__(self, data: List[Dict], tokenizer, intent2id: Dict, slot2id: Dict, max_length: int = 128):
        self.data = data
        self.tokenizer = tokenizer
        self.intent2id = intent2id
        self.slot2id = slot2id
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        tokens, intent, bio_tags = item['tokens'], item['intent'], item['bio_tags']

        encoding = self.tokenizer(
            tokens,
            is_split_into_words=True,
            padding='max_length',
            truncation=True,
            max_length=self.max_length,
            return_tensors='pt'
        )
        word_ids = encoding.word_ids(0)

        aligned_labels = []
        prev_word_idx = None
        for word_idx in word_ids:
            if word_idx is None:
                aligned_labels.append(self.slot2id['O'])
            elif word_idx != prev_word_idx:
                aligned_labels.append(self.slot2id[bio_tags[word_idx]])
            else:
                label = bio_tags[word_idx]
                aligned_labels.append(self.slot2id['I-' + label[2:]] if label.startswith('B-') else self.slot2id[label])
            prev_word_idx = word_idx

        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'intent_id': torch.tensor(self.intent2id[intent], dtype=torch.long),
            'slot_ids': torch.tensor(aligned_labels, dtype=torch.long)
        }


class TransformerJointSLU(nn.Module):
    def __init__(self, model_name: str, num_intents: int, num_slots: int, dropout: float = 0.1):
        super().__init__()
        self.transformer = AutoModel.from_pretrained(model_name)
        hidden_size = self.transformer.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.intent_classifier = nn.Linear(hidden_size, num_intents)
        self.slot_classifier = nn.Linear(hidden_size, num_slots)

    def forward(self, input_ids, attention_mask):
        outputs = self.transformer(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = self.dropout(outputs.last_hidden_state)
        pooled_output = self.dropout(outputs.pooler_output if hasattr(outputs, 'pooler_output') else sequence_output[:, 0])
        return self.intent_classifier(pooled_output), self.slot_classifier(sequence_output)

# %%
# Cell 4 — Loss, training and eval helpers
def compute_loss(intent_logits, slot_logits, intent_ids, slot_ids, attention_mask, intent_criterion, slot_criterion):
    intent_loss = intent_criterion(intent_logits, intent_ids)
    active_loss = attention_mask.view(-1) == 1
    active_logits = slot_logits.view(-1, slot_logits.size(-1))[active_loss]
    active_labels = slot_ids.view(-1)[active_loss]
    slot_loss = slot_criterion(active_logits, active_labels)
    return intent_loss + slot_loss, intent_loss, slot_loss


def train_epoch(model, dataloader, optimizer, scheduler, intent_criterion, slot_criterion, device):
    model.train()
    total_loss = total_intent_loss = total_slot_loss = 0

    for batch in tqdm(dataloader, desc="Training"):
        optimizer.zero_grad()
        input_ids, attention_mask = batch['input_ids'].to(device), batch['attention_mask'].to(device)
        intent_ids, slot_ids = batch['intent_id'].to(device), batch['slot_ids'].to(device)

        intent_logits, slot_logits = model(input_ids, attention_mask)
        loss, intent_loss, slot_loss = compute_loss(intent_logits, slot_logits, intent_ids, slot_ids, attention_mask, intent_criterion, slot_criterion)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        total_intent_loss += intent_loss.item()
        total_slot_loss += slot_loss.item()

    n = len(dataloader)
    return total_loss/n, total_intent_loss/n, total_slot_loss/n


def evaluate(model, dataloader, intent_criterion, slot_criterion, device, id2intent, id2slot):
    model.eval()
    total_loss = total_intent_loss = total_slot_loss = 0

    all_intent_preds, all_intent_trues = [], []
    all_slot_preds, all_slot_trues = [], []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating"):
            input_ids, attention_mask = batch['input_ids'].to(device), batch['attention_mask'].to(device)
            intent_ids, slot_ids = batch['intent_id'].to(device), batch['slot_ids'].to(device)

            intent_logits, slot_logits = model(input_ids, attention_mask)
            loss, intent_loss, slot_loss = compute_loss(intent_logits, slot_logits, intent_ids, slot_ids, attention_mask, intent_criterion, slot_criterion)

            total_loss += loss.item()
            total_intent_loss += intent_loss.item()
            total_slot_loss += slot_loss.item()

            intent_preds = torch.argmax(intent_logits, dim=1)
            all_intent_preds.extend(intent_preds.cpu().numpy())
            all_intent_trues.extend(intent_ids.cpu().numpy())

            slot_preds = torch.argmax(slot_logits, dim=2)
            for i in range(input_ids.size(0)):
                mask = attention_mask[i].cpu().numpy() == 1
                pred_tags = [id2slot[s.item()] for s in slot_preds[i][mask] if s.item() in id2slot]
                true_tags = [id2slot[s.item()] for s in slot_ids[i][mask] if s.item() in id2slot]
                all_slot_preds.append(pred_tags)
                all_slot_trues.append(true_tags)

    n = len(dataloader)
    intent_acc = accuracy_score(all_intent_trues, all_intent_preds)
    intent_f1 = f1_score(all_intent_trues, all_intent_preds, average='macro')
    slot_f1 = seq_f1_score(all_slot_trues, all_slot_preds)

    return (total_loss/n, total_intent_loss/n, total_slot_loss/n,
            intent_acc, intent_f1, slot_f1,
            all_intent_preds, all_intent_trues,
            all_slot_preds, all_slot_trues)

# %%
# Cell 5 — Helpers and main run

def load_json(path):
    with open(path, 'r') as f:
        return json.load(f)


def main():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    train_data, val_data, test_data = map(load_json, [
        'data/processed/train.json',
        'data/processed/val.json',
        'data/processed/test.json'
    ])
    metadata = load_json('data/processed/metadata.json')

    intent2id = {intent: i for i, intent in enumerate(metadata['intents'])}
    id2intent = {i: intent for intent, i in intent2id.items()}
    slot2id = {slot: i for i, slot in enumerate(metadata['slots'])}
    id2slot = {i: slot for slot, i in slot2id.items()}

    tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
    train_loader = DataLoader(TransformerSLUDataset(train_data, tokenizer, intent2id, slot2id), batch_size=16, shuffle=True)
    val_loader = DataLoader(TransformerSLUDataset(val_data, tokenizer, intent2id, slot2id), batch_size=16, shuffle=False)
    test_loader = DataLoader(TransformerSLUDataset(test_data, tokenizer, intent2id, slot2id), batch_size=16, shuffle=False)

    model = TransformerJointSLU("bert-base-uncased", len(intent2id), len(slot2id), dropout=0.1).to(device)
    optimizer = AdamW(model.parameters(), lr=2e-5)
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1 * len(train_loader) * 10), num_training_steps=len(train_loader) * 10)
    intent_criterion = nn.CrossEntropyLoss()
    slot_criterion = nn.CrossEntropyLoss()

    best_val_acc = 0
    for epoch in range(5):
        print(f"\nEpoch {epoch+1}/5")
        train_loss, train_intent_loss, train_slot_loss = train_epoch(model, train_loader, optimizer, scheduler, intent_criterion, slot_criterion, device)
        val_metrics = evaluate(model, val_loader, intent_criterion, slot_criterion, device, id2intent, id2slot)
        val_loss, val_intent_loss, val_slot_loss, val_acc, val_intent_f1, val_slot_f1, *_ = val_metrics

        print(f"Train Loss: {train_loss:.4f} (Intent: {train_intent_loss:.4f}, Slot: {train_slot_loss:.4f})")
        print(f"Val Loss: {val_loss:.4f} (Intent: {val_intent_loss:.4f}, Slot: {val_slot_loss:.4f})")
        print(f"Val Intent Acc: {val_acc:.4f}, Val Intent F1: {val_intent_f1:.4f}, Val Slot F1: {val_slot_f1:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), 'transformer_joint_model.pt')
            print(f"Model saved! Best Val Accuracy: {best_val_acc:.4f}")

    model.load_state_dict(torch.load('transformer_joint_model.pt'))
    test_metrics = evaluate(model, test_loader, intent_criterion, slot_criterion, device, id2intent, id2slot)
    test_loss, test_intent_loss, test_slot_loss, test_acc, test_intent_f1, test_slot_f1, test_int_preds, test_int_trues, test_slot_preds, test_slot_trues = test_metrics

    print(f"\nTest Results:\nLoss: {test_loss:.4f} (Intent: {test_intent_loss:.4f}, Slot: {test_slot_loss:.4f})")
    print(f"Intent Accuracy: {test_acc:.4f}, Intent F1: {test_intent_f1:.4f}, Slot F1: {test_slot_f1:.4f}")
    print("\nIntent Classification Report:")
    print(classification_report([id2intent[i] for i in test_int_trues],
                                [id2intent[i] for i in test_int_preds]))
    print("\nSlot Classification Report:")
    print(seq_classification_report(test_slot_trues, test_slot_preds))
    print("\nTraining complete!")


if __name__ == "__main__":
    main()

Using device: cuda

Epoch 1/5


Training:  40%|███▉      | 461/1167 [06:47<10:23,  1.13it/s]


KeyboardInterrupt: 

In [1]:
# single-cell fixed script with tokenization cache and clearer progress
import os
import re
import json
import random
from typing import List, Dict, Tuple
from collections import Counter
import chardet
import math
import time

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.metrics import accuracy_score, f1_score, classification_report
from seqeval.metrics import f1_score as seq_f1_score, classification_report as seq_classification_report
from tqdm import tqdm
from torch.cuda.amp import autocast, GradScaler

# ---------- (TranscriptPreprocessor same as before; abbreviated here) ----------
class TranscriptPreprocessor:
    def __init__(self, transcripts_dir: str, processed_dir: str):
        self.transcripts_dir = transcripts_dir
        self.processed_dir = processed_dir
        self.intents = set()
        self.slot_types = set()
        self.medical_keywords = {
            'SYMPTOM': ['pain', 'cough', 'breathless', 'itching', 'nausea', 'vomiting', 'fever',
                        'headache', 'dizziness', 'fatigue', 'swelling', 'rash', 'discharge',
                        'bleeding', 'numbness', 'tingling', 'weakness', 'stiffness'],
            'LOCATION': ['chest', 'abdomen', 'back', 'head', 'neck', 'arm', 'leg', 'knee',
                         'shoulder', 'hip', 'ankle', 'wrist', 'upper', 'lower', 'left', 'right'],
            'DURATION': ['days', 'weeks', 'months', 'years', 'hours', 'minutes', 'ago',
                         'since', 'yesterday', 'morning', 'evening', 'night', 'recently'],
            'SEVERITY': ['severe', 'mild', 'moderate', 'intense', 'slight', 'terrible',
                         'bad', 'worse', 'better', 'improved'],
            'PROGRESSION': ['getting worse', 'improving', 'constant', 'intermittent',
                            'comes and goes', 'fluctuates', 'stable'],
            'MEDICATION': ['ibuprofen', 'paracetamol', 'aspirin', 'antacid', 'antibiotic',
                           'inhaler', 'cream', 'ointment', 'tablet', 'pill'],
            'HISTORY': ['diabetes', 'hypertension', 'asthma', 'allergy', 'surgery',
                        'family history', 'previous', 'diagnosed']
        }
        os.makedirs(processed_dir, exist_ok=True)

    def load_transcripts(self) -> List[Dict]:
        transcripts = []
        for filename in os.listdir(self.transcripts_dir):
            if filename.endswith('.txt'):
                filepath = os.path.join(self.transcripts_dir, filename)
                category = filename[:3]
                transcript_id = filename[:-4]
                with open(filepath, 'rb') as f:
                    encoding = chardet.detect(f.read())['encoding'] or 'utf-8'
                with open(filepath, 'r', encoding=encoding, errors='ignore') as f:
                    content = f.read()
                transcripts.append({'id': transcript_id, 'category': category, 'content': content, 'filepath': filepath})
        return transcripts

    def parse_turns(self, content: str) -> List[Dict]:
        turns = []
        for line in content.strip().split('\n'):
            line = line.strip()
            if not line:
                continue
            if line.startswith('D:'):
                turns.append({'speaker': 'Doctor', 'text': line[2:].strip()})
            elif line.startswith('P:'):
                turns.append({'speaker': 'Patient', 'text': line[2:].strip()})
        return turns

    def normalize_text(self, text: str) -> str:
        return re.sub(r'\s+', ' ', text.lower().strip())

    def tokenize(self, text: str) -> List[str]:
        return re.sub(r'([?.!,;])', r' \1 ', text).split()

    def infer_intent(self, speaker: str, text: str) -> str:
        t = text.lower()
        if speaker == 'Doctor':
            if any(q in t for q in ['what', 'when', 'where', 'how', 'which', 'who']):
                if 'symptom' in t or 'feel' in t or 'problem' in t:
                    return 'Ask_Symptom'
                elif any(word in t for word in ['long', 'when', 'start', 'began']):
                    return 'Ask_Duration'
                elif 'where' in t or 'location' in t:
                    return 'Ask_Location'
                elif any(word in t for word in ['severe', 'bad', 'intense']):
                    return 'Ask_Severity'
                elif 'medication' in t or 'medicine' in t or 'taking' in t:
                    return 'Ask_Medication'
                elif 'history' in t or 'before' in t or 'previous' in t:
                    return 'Ask_History'
                else:
                    return 'Ask_General'
            elif any(word in t for word in ['test', 'examine', 'check']):
                return 'Propose_Examination'
            elif any(word in t for word in ['prescribe', 'recommend', 'suggest']):
                return 'Provide_Treatment'
            return 'Doctor_Other'
        else:
            for slot, keywords in self.medical_keywords.items():
                if any(word in t for word in keywords):
                    return f'Provide_{slot.capitalize()}'
            return 'Patient_Other'

    def extract_slots(self, text: str) -> List[Dict]:
        t = text.lower()
        entities = []
        for slot_type, keywords in self.medical_keywords.items():
            for kw in keywords:
                for match in re.finditer(rf'\b{re.escape(kw)}\w*\b', t):
                    entities.append({'type': slot_type, 'text': match.group(), 'start': match.start(), 'end': match.end()})
        entities.sort(key=lambda x: x['start'])
        non_overlap = []
        for e in entities:
            if not non_overlap or e['start'] >= non_overlap[-1]['end']:
                non_overlap.append(e)
        return non_overlap

    def tokens_to_bio(self, tokens: List[str], text: str, entities: List[Dict]) -> List[str]:
        bio = ['O'] * len(tokens)
        char_to_token = []
        pos = 0
        text_lower = text.lower()
        for i, token in enumerate(tokens):
            start = text_lower.find(token, pos)
            start = start if start != -1 else pos
            end = start + len(token)
            char_to_token.append((start, end, i))
            pos = end
        for e in entities:
            overlap = [i for s, en, i in char_to_token if s < e['end'] and en > e['start']]
            if overlap:
                bio[overlap[0]] = f'B-{e["type"]}'
                for idx in overlap[1:]:
                    bio[idx] = f'I-{e["type"]}'
        return bio

    def process_transcripts(self) -> List[Dict]:
        data = []
        for t in self.load_transcripts():
            for idx, turn in enumerate(self.parse_turns(t['content'])):
                text = turn['text']
                norm_text = self.normalize_text(text)
                tokens = self.tokenize(norm_text)
                intent = self.infer_intent(turn['speaker'], text)
                self.intents.add(intent)
                entities = self.extract_slots(norm_text)
                self.slot_types.update([e['type'] for e in entities])
                bio_tags = self.tokens_to_bio(tokens, norm_text, entities)
                data.append({'transcript_id': t['id'], 'category': t['category'], 'turn_id': idx,
                             'speaker': turn['speaker'], 'original_text': text, 'normalized_text': norm_text,
                             'tokens': tokens, 'intent': intent, 'entities': entities, 'bio_tags': bio_tags})
        return data

    def create_splits(self, data: List[Dict], train_ratio=0.7, val_ratio=0.15, test_ratio=0.15, seed=42) -> Tuple[List[Dict], List[Dict], List[Dict]]:
        ids = list({item['transcript_id'] for item in data})
        random.seed(seed)
        random.shuffle(ids)
        n_train = int(len(ids) * train_ratio)
        n_val = int(len(ids) * val_ratio)
        train_ids, val_ids, test_ids = set(ids[:n_train]), set(ids[n_train:n_train+n_val]), set(ids[n_train+n_val:])
        train_data = [x for x in data if x['transcript_id'] in train_ids]
        val_data = [x for x in data if x['transcript_id'] in val_ids]
        test_data = [x for x in data if x['transcript_id'] in test_ids]
        return train_data, val_data, test_data

    def save_processed_data(self, train_data: List[Dict], val_data: List[Dict], test_data: List[Dict]):
        for name, d in zip(['train', 'val', 'test'], [train_data, val_data, test_data]):
            with open(os.path.join(self.processed_dir, f'{name}.json'), 'w') as f:
                json.dump(d, f, indent=2)
        metadata = {'intents': sorted(list(self.intents)),
                    'slots': ['O'] + [f'{p}-{s}' for s in sorted(self.slot_types) for p in ['B', 'I']],
                    'num_intents': len(self.intents),
                    'num_slots': len(self.slot_types)+1,
                    'train_size': len(train_data),
                    'val_size': len(val_data),
                    'test_size': len(test_data)}
        with open(os.path.join(self.processed_dir, 'metadata.json'), 'w') as f:
            json.dump(metadata, f, indent=2)
        return metadata

    def get_statistics(self, data: List[Dict]) -> Dict:
        return {
            'num_utterances': len(data),
            'num_transcripts': len(set([x['transcript_id'] for x in data])),
            'intent_distribution': Counter([x['intent'] for x in data]),
            'category_distribution': Counter([x['category'] for x in data]),
            'speaker_distribution': Counter([x['speaker'] for x in data]),
            'avg_tokens_per_utterance': sum(len(x['tokens']) for x in data)/len(data) if data else 0,
            'slot_type_counts': Counter([e['type'] for x in data for e in x['entities']])
        }

# ---------- Dataset with tokenization cache ----------
class TransformerSLUDataset(Dataset):
    """
    Pre-tokenizes and aligns BIO labels. Uses a disk cache to avoid repeated pre-tokenization delays.
    """
    def __init__(self, data: List[Dict], tokenizer, intent2id: Dict, slot2id: Dict, max_length: int = 128, cache_path: str = None):
        self.data = data
        self.tokenizer = tokenizer
        self.intent2id = intent2id
        self.slot2id = slot2id
        self.max_length = max_length
        self.cache_path = cache_path

        if self.cache_path and os.path.exists(self.cache_path):
            # Load cached pre-tokenized encodings (fast)
            print(f"Loading tokenized cache from {self.cache_path}")
            cached = torch.load(self.cache_path)
            # Validate basic shape if possible
            self.encodings = cached
            print(f"Loaded {len(self.encodings)} cached samples.")
            return

        # Otherwise pre-tokenize and save to cache (this may take some time on first run)
        self.encodings = []
        print("Pre-tokenizing dataset (this runs only once and will be cached)...")
        for item in tqdm(self.data, desc="Pre-tokenizing dataset", leave=True):
            tokens = item['tokens']
            bio_tags = item['bio_tags']
            enc = self.tokenizer(
                tokens,
                is_split_into_words=True,
                padding='max_length',
                truncation=True,
                max_length=self.max_length,
                return_attention_mask=True
            )
            try:
                word_ids = enc.word_ids()
            except Exception:
                # fallback: mark all as None (rare for fast tokenizer)
                word_ids = [None] * len(enc['input_ids'])

            aligned = []
            prev_word = None
            for wid in word_ids:
                if wid is None:
                    aligned.append(self.slot2id.get('O', 0))
                elif wid != prev_word:
                    aligned.append(self.slot2id.get(bio_tags[wid], self.slot2id.get('O', 0)))
                else:
                    lab = bio_tags[wid]
                    if lab.startswith('B-'):
                        lab = 'I-' + lab[2:]
                    aligned.append(self.slot2id.get(lab, self.slot2id.get('O', 0)))
                prev_word = wid

            # pad/truncate
            if len(aligned) < self.max_length:
                aligned += [self.slot2id.get('O', 0)] * (self.max_length - len(aligned))
            else:
                aligned = aligned[:self.max_length]

            self.encodings.append({
                'input_ids': torch.tensor(enc['input_ids'], dtype=torch.long),
                'attention_mask': torch.tensor(enc['attention_mask'], dtype=torch.long),
                'intent_id': torch.tensor(self.intent2id[item['intent']], dtype=torch.long),
                'slot_ids': torch.tensor(aligned, dtype=torch.long)
            })

        if self.cache_path:
            # Save cache
            try:
                torch.save(self.encodings, self.cache_path)
                print(f"Saved tokenized cache to {self.cache_path}")
            except Exception as e:
                print("Warning: failed to save tokenized cache:", e)

    def __len__(self):
        return len(self.encodings)

    def __getitem__(self, idx):
        return self.encodings[idx]


def collate_fn(batch):
    input_ids = torch.stack([b['input_ids'] for b in batch], dim=0)
    attention_mask = torch.stack([b['attention_mask'] for b in batch], dim=0)
    intent_id = torch.stack([b['intent_id'] for b in batch], dim=0)
    slot_ids = torch.stack([b['slot_ids'] for b in batch], dim=0)
    return {
        'input_ids': input_ids,
        'attention_mask': attention_mask,
        'intent_id': intent_id,
        'slot_ids': slot_ids
    }

# ---------- Model ----------
class TransformerJointSLU(nn.Module):
    def __init__(self, model_name: str, num_intents: int, num_slots: int, dropout: float = 0.1):
        super().__init__()
        self.transformer = AutoModel.from_pretrained(model_name)
        hidden_size = self.transformer.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.intent_classifier = nn.Linear(hidden_size, num_intents)
        self.slot_classifier = nn.Linear(hidden_size, num_slots)

    def forward(self, input_ids, attention_mask):
        outputs = self.transformer(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = self.dropout(outputs.last_hidden_state)
        pooled_output = self.dropout(outputs.pooler_output if hasattr(outputs, 'pooler_output') else sequence_output[:, 0])
        return self.intent_classifier(pooled_output), self.slot_classifier(sequence_output)


# ---------- Loss + train/eval (AMP) ----------
def compute_loss(intent_logits, slot_logits, intent_ids, slot_ids, attention_mask, intent_criterion, slot_criterion):
    intent_loss = intent_criterion(intent_logits, intent_ids)
    active_loss = attention_mask.view(-1) == 1
    active_logits = slot_logits.view(-1, slot_logits.size(-1))[active_loss]
    active_labels = slot_ids.view(-1)[active_loss]
    slot_loss = slot_criterion(active_logits, active_labels)
    return intent_loss + slot_loss, intent_loss, slot_loss


def train_epoch(model, dataloader, optimizer, scheduler, intent_criterion, slot_criterion, device, scaler: GradScaler = None):
    model.train()
    total_loss = total_intent_loss = total_slot_loss = 0.0

    # show outer progress for epoch; per-batch tqdm inside gives percent
    for batch in tqdm(dataloader, desc="Training (batches)", leave=False):
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device, non_blocking=True)
        attention_mask = batch['attention_mask'].to(device, non_blocking=True)
        intent_ids = batch['intent_id'].to(device, non_blocking=True)
        slot_ids = batch['slot_ids'].to(device, non_blocking=True)

        with autocast(enabled=(device.type == 'cuda')):
            intent_logits, slot_logits = model(input_ids, attention_mask)
            loss, intent_loss, slot_loss = compute_loss(intent_logits, slot_logits, intent_ids, slot_ids, attention_mask, intent_criterion, slot_criterion)

        if scaler is not None:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        scheduler.step()

        total_loss += loss.item()
        total_intent_loss += intent_loss.item()
        total_slot_loss += slot_loss.item()

    n = len(dataloader) if len(dataloader) > 0 else 1
    return total_loss/n, total_intent_loss/n, total_slot_loss/n


def evaluate(model, dataloader, intent_criterion, slot_criterion, device, id2intent, id2slot):
    model.eval()
    total_loss = total_intent_loss = total_slot_loss = 0.0

    all_intent_preds, all_intent_trues = [], []
    all_slot_preds, all_slot_trues = [], []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating (batches)", leave=False):
            input_ids = batch['input_ids'].to(device, non_blocking=True)
            attention_mask = batch['attention_mask'].to(device, non_blocking=True)
            intent_ids = batch['intent_id'].to(device, non_blocking=True)
            slot_ids = batch['slot_ids'].to(device, non_blocking=True)

            with autocast(enabled=(device.type == 'cuda')):
                intent_logits, slot_logits = model(input_ids, attention_mask)
                loss, intent_loss, slot_loss = compute_loss(intent_logits, slot_logits, intent_ids, slot_ids, attention_mask, intent_criterion, slot_criterion)

            total_loss += loss.item()
            total_intent_loss += intent_loss.item()
            total_slot_loss += slot_loss.item()

            intent_preds = torch.argmax(intent_logits, dim=1)
            all_intent_preds.extend(intent_preds.cpu().numpy())
            all_intent_trues.extend(intent_ids.cpu().numpy())

            slot_preds = torch.argmax(slot_logits, dim=2).cpu().numpy()
            attn = attention_mask.cpu().numpy()
            slot_trues = slot_ids.cpu().numpy()

            for i in range(input_ids.size(0)):
                mask = attn[i] == 1
                pred_tags = [id2slot[int(s)] for s, m in zip(slot_preds[i], mask) if m and int(s) in id2slot]
                true_tags = [id2slot[int(s)] for s, m in zip(slot_trues[i], mask) if m and int(s) in id2slot]
                all_slot_preds.append(pred_tags)
                all_slot_trues.append(true_tags)

    n = len(dataloader) if len(dataloader) > 0 else 1
    intent_acc = accuracy_score(all_intent_trues, all_intent_preds) if all_intent_trues else 0.0
    intent_f1 = f1_score(all_intent_trues, all_intent_preds, average='macro') if all_intent_trues else 0.0
    slot_f1 = seq_f1_score(all_slot_trues, all_slot_preds) if all_slot_trues else 0.0
    return (total_loss/n, total_intent_loss/n, total_slot_loss/n,
            intent_acc, intent_f1, slot_f1,
            all_intent_preds, all_intent_trues,
            all_slot_preds, all_slot_trues)


def load_json(path):
    with open(path, 'r') as f:
        return json.load(f)

# ---------- MAIN ----------
def main():
    DATA_DIR = 'data/processed'  # must contain train.json, val.json, test.json, metadata.json
    MODEL_NAME = "bert-base-uncased"

    # SAFER defaults for Windows / Notebook environment
    BATCH_SIZE = 8          # smaller to avoid OOM / swapping
    MAX_LENGTH = 128
    EPOCHS = 5
    # IMPORTANT: use 0 workers in notebooks/Windows to avoid spawn/hang issues
    NUM_WORKERS = 0
    CACHE_PATH = os.path.join(DATA_DIR, 'tokenized_cache.pt')

    # Disable tokenizer parallelism (avoids weird hangs)
    os.environ["TOKENIZERS_PARALLELISM"] = "false"

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Device: {device} | workers: {NUM_WORKERS} | batch: {BATCH_SIZE} | pin_memory: False")

    # Load JSON data
    train_data, val_data, test_data = map(load_json, [
        os.path.join(DATA_DIR, 'train.json'),
        os.path.join(DATA_DIR, 'val.json'),
        os.path.join(DATA_DIR, 'test.json')
    ])
    metadata = load_json(os.path.join(DATA_DIR, 'metadata.json'))

    intent2id = {intent: i for i, intent in enumerate(metadata['intents'])}
    id2intent = {i: intent for intent, i in intent2id.items()}
    slot2id = {slot: i for i, slot in enumerate(metadata['slots'])}
    id2slot = {i: slot for slot, i in slot2id.items()}

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

    # Separate cache files per split (as before)
    train_cache = CACHE_PATH.replace('.pt', '.train.pt')
    val_cache = CACHE_PATH.replace('.pt', '.val.pt')
    test_cache = CACHE_PATH.replace('.pt', '.test.pt')

    # Create datasets (loads caches if present)
    train_dataset = TransformerSLUDataset(train_data, tokenizer, intent2id, slot2id, max_length=MAX_LENGTH, cache_path=train_cache)
    val_dataset = TransformerSLUDataset(val_data, tokenizer, intent2id, slot2id, max_length=MAX_LENGTH, cache_path=val_cache)
    test_dataset = TransformerSLUDataset(test_data, tokenizer, intent2id, slot2id, max_length=MAX_LENGTH, cache_path=test_cache)

    print(f"Dataset sizes -> train: {len(train_dataset)}, val: {len(val_dataset)}, test: {len(test_dataset)}")
    if len(train_dataset) == 0:
        raise RuntimeError("Train dataset is empty — check your processed JSON files.")

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=False, collate_fn=collate_fn)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=False, collate_fn=collate_fn)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=False, collate_fn=collate_fn)

    # Model & optimizer
    model = TransformerJointSLU(MODEL_NAME, len(intent2id), len(slot2id), dropout=0.1).to(device)

    # Temporarily skip gradient checkpointing to avoid extra overhead/hangs
    # try:
    #     model.transformer.gradient_checkpointing_enable()
    #     print("Gradient checkpointing enabled.")
    # except Exception:
    #     pass

    optimizer = AdamW(model.parameters(), lr=2e-5)
    total_steps = max(1, len(train_loader) * EPOCHS)
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps)

    intent_criterion = nn.CrossEntropyLoss()
    slot_criterion = nn.CrossEntropyLoss()
    scaler = GradScaler() if device.type == 'cuda' else None

    # ---- SMOKE TEST: run a few batches forward/back (no grad) to confirm everything works and prints timing ----
    model.eval()
    try:
        it = iter(train_loader)
        print("\nRunning quick smoke test on 3 batches to verify data/model/GPU...")
        for i in range(3):
            batch = next(it)
            input_ids = batch['input_ids'].to(device, non_blocking=True)
            attention_mask = batch['attention_mask'].to(device, non_blocking=True)
            t0 = torch.cuda.Event(enable_timing=True) if device.type=='cuda' else None
            t1 = torch.cuda.Event(enable_timing=True) if device.type=='cuda' else None
            if device.type == 'cuda':
                t0.record()
            with torch.no_grad():
                with autocast(enabled=(device.type == 'cuda')):
                    intent_logits, slot_logits = model(input_ids, attention_mask)
            if device.type == 'cuda':
                t1.record()
                torch.cuda.synchronize()
                ms = t0.elapsed_time(t1)
                print(f"  Batch {i+1} forward time: {ms:.1f} ms")
            else:
                print(f"  Batch {i+1} forward done (CPU).")
        print("Smoke test passed.\n")
    except StopIteration:
        print("Smoke test: not enough batches to run check.")
    except RuntimeError as e:
        print("RuntimeError during smoke test — likely OOM or device error:", e)
        print("Try reducing BATCH_SIZE or checking GPU memory.")
        return

    # TRAINING LOOP: You should now see epoch / batch progress advance
    best_val_acc = 0.0
    for epoch in range(EPOCHS):
        print(f"\n=== Epoch {epoch+1}/{EPOCHS} ===")
        train_loss, train_intent_loss, train_slot_loss = train_epoch(model, train_loader, optimizer, scheduler, intent_criterion, slot_criterion, device, scaler)
        val_metrics = evaluate(model, val_loader, intent_criterion, slot_criterion, device, id2intent, id2slot)
        val_loss, val_intent_loss, val_slot_loss, val_acc, val_intent_f1, val_slot_f1, *_ = val_metrics

        print(f"Train Loss: {train_loss:.4f} | Val Intent Acc: {val_acc:.4f} | Val Slot F1: {val_slot_f1:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), 'transformer_joint_model.pt')
            print(f"Saved best model (val_acc={best_val_acc:.4f})")

    # Final test evaluation
    if os.path.exists('transformer_joint_model.pt'):
        model.load_state_dict(torch.load('transformer_joint_model.pt', map_location=device))
    test_metrics = evaluate(model, test_loader, intent_criterion, slot_criterion, device, id2intent, id2slot)
    test_loss, test_intent_loss, test_slot_loss, test_acc, test_intent_f1, test_slot_f1, test_int_preds, test_int_trues, test_slot_preds, test_slot_trues = test_metrics

    print("\n=== Test Results ===")
    print(f"Loss: {test_loss:.4f} | Intent Acc: {test_acc:.4f} | Slot F1: {test_slot_f1:.4f}")
    print("Done.")


if __name__ == "__main__":
    main()


Device: cuda | workers: 0 | batch: 8 | pin_memory: False
Loading tokenized cache from data/processed\tokenized_cache.train.pt
Loaded 18657 cached samples.
Loading tokenized cache from data/processed\tokenized_cache.val.pt
Loaded 3872 cached samples.
Loading tokenized cache from data/processed\tokenized_cache.test.pt
Loaded 4075 cached samples.
Dataset sizes -> train: 18657, val: 3872, test: 4075


C:\Users\Raghu Vamsi\AppData\Local\Temp\ipykernel_25372\3990065474.py:476: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler() if device.type == 'cuda' else None
C:\Users\Raghu Vamsi\AppData\Local\Temp\ipykernel_25372\3990065474.py:492: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=(device.type == 'cuda')):



Running quick smoke test on 3 batches to verify data/model/GPU...
  Batch 1 forward time: 602.7 ms
  Batch 2 forward time: 24.9 ms
  Batch 3 forward time: 24.5 ms
Smoke test passed.


=== Epoch 1/5 ===


Training (batches):   0%|          | 0/2333 [00:00<?, ?it/s]C:\Users\Raghu Vamsi\AppData\Local\Temp\ipykernel_25372\3990065474.py:332: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=(device.type == 'cuda')):
Evaluating (batches):   0%|          | 0/484 [00:00<?, ?it/s]          C:\Users\Raghu Vamsi\AppData\Local\Temp\ipykernel_25372\3990065474.py:371: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=(device.type == 'cuda')):


Train Loss: 0.9377 | Val Intent Acc: 0.9522 | Val Slot F1: 0.9442
Saved best model (val_acc=0.9522)

=== Epoch 2/5 ===


...

Train Loss: 0.0555 | Val Intent Acc: 0.9832 | Val Slot F1: 0.9971
Saved best model (val_acc=0.9832)

=== Epoch 5/5 ===


Training (batches):   0%|          | 0/2333 [00:00<?, ?it/s]C:\Users\Raghu Vamsi\AppData\Local\Temp\ipykernel_25372\3990065474.py:332: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=(device.type == 'cuda')):
Evaluating (batches):   0%|          | 0/484 [00:00<?, ?it/s]          C:\Users\Raghu Vamsi\AppData\Local\Temp\ipykernel_25372\3990065474.py:371: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=(device.type == 'cuda')):


Train Loss: 0.0405 | Val Intent Acc: 0.9850 | Val Slot F1: 0.9971
Saved best model (val_acc=0.9850)


Evaluating (batches):   0%|          | 0/510 [00:00<?, ?it/s]C:\Users\Raghu Vamsi\AppData\Local\Temp\ipykernel_25372\3990065474.py:371: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=(device.type == 'cuda')):



=== Test Results ===
Loss: 0.0938 | Intent Acc: 0.9845 | Slot F1: 0.9955
Done.


In [8]:
import pprint
import tempfile, shutil, re
from collections import defaultdict, Counter
from pathlib import Path

# Simple duration and severity helpers
_DURATION_RE = re.compile(r'\b(?:about\s+)?(\d+\s*(?:days?|weeks?|months?|years?|hours?|minutes?|hrs|min)|a\s+(?:day|week|month|year))\b', flags=re.I)
_SEVERITY_SCORE = {'terrible':5,'severe':5,'worse':4,'bad':4,'intense':4,'moderate':3,'mild':2,'slight':1,'improved':0,'better':0}

def _first(iterable, default=None):
    for x in iterable:
        return x
    return default

def _extract_first_duration(text):
    m = _DURATION_RE.search(text)
    return m.group(1).lower() if m else None

def _pick_best_severity(sev_list):
    best = None; best_score = -1
    for s in sev_list:
        score = _SEVERITY_SCORE.get(s.lower(), 0)
        if score > best_score:
            best_score = score; best = s
    return best

def _uniq_preserve_order(seq):
    seen = set(); out = []
    for x in seq:
        if x not in seen:
            seen.add(x); out.append(x)
    return out

def process_file_to_tiny_summary(txt_path: str, write_json: str = None):
    """
    Ultra-compact summary for a single transcript .txt file.
    Returns: {'file':..., 'transcript_summaries':[ {transcript_id, category, num_turns,
                                                   top_symptoms, main_location, main_duration,
                                                   severity, progression, intents_count,
                                                   example_utterance} ] }
    """
    txt_path = Path(txt_path)
    if not txt_path.exists():
        raise FileNotFoundError(txt_path)

    tmpdir = Path(tempfile.mkdtemp(prefix="single_tx_"))
    try:
        shutil.copy2(txt_path, tmpdir / txt_path.name)
        tp = TranscriptPreprocessor(str(tmpdir), processed_dir=str(tmpdir))
        turns = tp.process_transcripts()
        if not turns:
            return {'file': str(txt_path), 'transcript_summaries': []}

        by_tid = defaultdict(list)
        for t in turns:
            by_tid[t['transcript_id']].append(t)

        summaries = []
        for tid, tlist in by_tid.items():
            symptoms = []
            locations = []
            durations = []
            severities = []
            progressions = []
            intents = []
            example = None

            for t in tlist:
                intents.append(t.get('intent'))
                norm = t.get('normalized_text','')
                # pick first patient utterance as example
                if example is None and t.get('speaker','').lower().startswith('p'):
                    example = (t.get('original_text') or t.get('normalized_text'))[:200]

                for ent in t.get('entities', []):
                    typ = ent.get('type'); txt = ent.get('text')
                    if not typ or not txt:
                        continue
                    if typ == 'SYMPTOM': symptoms.append(txt)
                    elif typ == 'LOCATION': locations.append(txt)
                    elif typ == 'DURATION': durations.append(txt)
                    elif typ == 'SEVERITY': severities.append(txt)
                    elif typ == 'PROGRESSION': progressions.append(txt)

                # fallback duration regex
                d = _extract_first_duration(norm)
                if d:
                    durations.append(d)

            top_symptoms = _uniq_preserve_order(symptoms)[:3]
            main_location = _first(_uniq_preserve_order(locations))
            main_duration = _first(durations)  # keep first found
            severity_choice = _pick_best_severity(severities)
            progression_choice = _first(_uniq_preserve_order(progressions))

            summary = {
                'transcript_id': tid,
                'category': tlist[0].get('category'),
                'num_turns': len(tlist),
                'top_symptoms': top_symptoms,
                'main_location': main_location,
                'main_duration': main_duration,
                'severity': severity_choice,
                'progression': progression_choice,
                'intents_count': dict(Counter(intents)),
                'example_utterance': example
            }
            summaries.append(summary)

        result = {'file': str(txt_path), 'transcript_summaries': summaries}
        if write_json:
            Path(write_json).parent.mkdir(parents=True, exist_ok=True)
            with open(write_json, 'w', encoding='utf-8') as f:
                json.dump(result, f, ensure_ascii=False, indent=2)
        return result

    finally:
        try: shutil.rmtree(tmpdir)
        except Exception: pass

# Example:
tiny = process_file_to_tiny_summary("sample_input.txt")
pprint.pprint(tiny)


{'file': 'sample_input.txt',
 'transcript_summaries': [{'category': 'sam',
                           'example_utterance': "I've had this pain in my back "
                                                'thats kind of like in my '
                                                'lower back and my buttocks '
                                                "and it's now radiating down "
                                                "to my right leg. I've had "
                                                "this for awhile now, but it's "
                                                'getting worse and I just want '
                                                'to c',
                           'intents_count': {'Ask_Duration': 1,
                                             'Ask_General': 3,
                                             'Ask_Location': 1,
                                             'Doctor_Other': 4,
                                             'Patient_Other':